# 第8章「併売パターンを発見する」— コード実行ノートブック

本ノートブックは `08_main.tex` に掲載されている Python コードを抽出・整理したものです。
第3節「実践：Aprioriアルゴリズムによるアソシエーションルールの発見」のコードが対象です。


In [1]:
import sys
from pathlib import Path

import matplotlib_fontja  # noqa: F401

for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path))
        break

    if (path / "src" / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path / "src"))
        break

from common.notebook_style import apply_dataframe_style

apply_dataframe_style()

# 第8章 併売パターンを発見する


## 第8章 第3節 第1項のコード

### データの読み込み

Kaggle でCC0として公開されている `Groceries data.csv` を Polars で読み込み、必要な3列を選んで行数と列数を確認する。

出典: Rashik Rahman, [Groceries dataset for Market Basket Analysis (MBA)](https://www.kaggle.com/datasets/rashikrahmanpritom/groceries-dataset-for-market-basket-analysismba)（データセット: Version 2、Kaggleでのライセンス表示: [CC0 1.0](https://creativecommons.org/publicdomain/zero/1.0/)、2026年9月13日確認）。

#### 処理内容

1. **ライブラリのインポート**
   高速なデータフレームライブラリ `polars` をインポートする。

2. **ファイルパスの指定**
   `./data/Groceries data.csv` をファイル名として変数 `file_name` に格納する。

3. **CSV ファイルの読み込み**
   `pl.read_csv()` でCSVを読み込み、会員番号、購入日、商品名の3列を選んで DataFrame として `df` に格納する。
   各行は1つの商品明細を表し、会員番号・購入日・商品名が記録されている。

4. **件数の確認**
   読み込んだデータの行数と列数を確認する。


In [3]:
import polars as pl

# ファイルの指定
file_name = "./data/Groceries data.csv"

# CSVを読み込み、使う3列を選択
df = pl.read_csv(file_name).select(["Member_number", "Date", "itemDescription"])

print(f"行数: {df.height}, 列数: {df.width}")

### トランザクション形式への整形

同じ会員・同じ購入日の商品明細を集約し、購入した商品をリストとしてまとめたトランザクション形式（リストのリスト）に変換する。

#### 処理内容

1. **グループ化と集約**
   `df.group_by(["Member_number", "Date"])` で会員番号と購入日をキーにグループ化し、
   `pl.col("itemDescription").unique(maintain_order=True)` で商品名の重複を除いてリストにまとめる。

2. **トランザクションのリストの抽出**
   `transactions_df["items"].to_list()` で商品のリストだけを取り出し、リストのリストに変換する。

3. **先頭データの確認**
   `transactions[:5]` で先頭5件のトランザクションを表示する。


In [4]:
# 同じ会員番号・同じ購入日の商品明細を1つの買い物として集約
transactions_df = df.group_by(
    ["Member_number", "Date"],
    maintain_order=True,
).agg(pl.col("itemDescription").unique(maintain_order=True).alias("items"))

# 商品のリストだけを取り出して、リストのリストに変換
transactions = transactions_df["items"].to_list()

# 先頭5件を表示
transactions[:5]

[['tropical fruit', 'rolls/buns', 'candy'],
 ['whole milk', 'tropical fruit', 'chocolate'],
 ['pip fruit', 'other vegetables', 'flour'],
 ['other vegetables', 'onions', 'shopping bags'],
 ['whole milk', 'other vegetables', 'white bread']]

## 第8章 第3節 第2項のコード

### ワンホットエンコーディング

トランザクションのリストを、各アイテムを列、各トランザクションを行とし、購入の有無をTrue/Falseで表したDataFrameに変換する（ワンホットエンコーディング）。

#### 処理内容

1. **商品名の一覧の取得**
   元のDataFrameの `itemDescription` 列から重複を除いた商品名の一覧を取得して並べ替え、`all_items` に格納する。

2. **ワンホットエンコーディング**
   各アイテムを列とし、各トランザクションにそのアイテムが含まれるかどうかをTrue/Falseで表した Polars DataFrame を作成する。

3. **表示する列の選択と確認**
   先頭のトランザクションに含まれる商品を優先した列の順番で先頭5行を表示する。


In [5]:
import polars as pl

# 元のDataFrameから商品名の一覧を取得
all_items = df["itemDescription"].unique().sort().to_list()

# 各アイテムを列に設定し、各トランザクションでアイテムの購入の有無をTrue/Falseで格納
# ワンホットエンコーディング処理
df = pl.DataFrame(
    {
        item: [item in transaction for transaction in transactions]
        for item in all_items
    }
)

preview_columns = (
    transactions[0]
    + [item for item in all_items if item not in transactions[0]][:4]
)
data_preview = df.slice(0, 5).select(preview_columns)
data_preview

tropical fruit,rolls/buns,candy,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener
bool,bool,bool,bool,bool,bool,bool
true,true,true,false,false,false,false
true,false,false,false,false,false,false
false,false,false,false,false,false,false
false,false,false,false,false,false,false
false,false,false,false,false,false,false


### Aprioriアルゴリズムの実行

mlxtend の `apriori` 関数を使って、支持度が最小支持度以上のアイテムセットを抽出する。

#### 処理内容

1. **apriori 関数の読み込み**
   `from mlxtend.frequent_patterns import apriori` で `apriori` 関数を読み込む。

2. **pandas DataFrame への変換**
   mlxtend は pandas DataFrame を入力として要求するため、`.to_pandas()` で変換する。

3. **頻出するアイテムセットの抽出**
   `apriori(df_pd, min_support=0.005, use_colnames=True)` で支持度が0.005以上のアイテムセットを抽出する。

4. **結果の確認**
   `frequent_itemsets.head()` で頻出するアイテムセットの先頭5件を表示する。


In [6]:
# apriori 関数の読み込み
from mlxtend.frequent_patterns import apriori

# mlxtend は pandas DataFrame を要求するため変換
df_pd = df.to_pandas()

# 最小支持度を満たすアイテムセットの抽出
frequent_itemsets = apriori(df_pd, min_support=0.005, use_colnames=True)
frequent_itemsets_preview = frequent_itemsets.head().round(4)
frequent_itemsets_preview

,support,itemsets
0,0.0214,frozenset({UHT-milk})
1,0.0081,frozenset({baking powder})
2,0.0340,frozenset({beef})
3,0.0218,frozenset({berries})
4,0.0166,frozenset({beverages})


## 第8章 第3節 第3項のコード

### アソシエーションルールの作成と評価

抽出したアイテムセットから、確信度・リフト値を使ってアソシエーションルールを生成・絞り込む。

#### 処理内容

1. **association_rules のインポート**
   `from mlxtend.frequent_patterns import association_rules` でルールを生成する関数を読み込む。

2. **ルールの生成**
   `association_rules()` に `metric="confidence"`, `min_threshold=0.02` を指定し、
   確信度0.02以上のルールをすべて生成する。

3. **リフト値による絞り込み**
   `.query("lift >= 1.0")` でリフト値が1.0以上のルールに絞り込む。

4. **使用する列の選択**
   `antecedents`, `consequents`, `support`, `confidence`, `lift` の5列を選択し、後で結果を比較するために `mlxtend_result_df` に保存する。

5. **結果の表示**
   `support`, `confidence`, `lift` を優先する順番で、それぞれ値が大きい順に並べ替え、小数点以下4桁で先頭5件を表示する。


In [7]:
from mlxtend.frequent_patterns import association_rules

min_threshold = 0.02

# 頻出するアイテムセットから確信度が0.02以上のルールを生成
rules_df = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=min_threshold,
)

# リフト値が1.0以上のルールに絞り込む
rules_df = rules_df.query("lift >= 1.0")

# 利用する列名の設定
columns = [
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
]

# 後で結果を比較するために保存
mlxtend_result_df = rules_df[columns].copy()

# support, confidence, lift を順に優先し、各値が大きい順に並べ替えて小数点以下4桁で表示
rules_preview = (
    rules_df.sort_values(by=["support", "confidence", "lift"], ascending=False)[
        columns
    ]
    .reset_index(drop=True)
    .round(4)
    .head()
)
rules_preview

,antecedents,consequents,support,confidence,lift
0,frozenset({sausage}),frozenset({soda}),0.0059,0.0986,1.0150
1,frozenset({soda}),frozenset({sausage}),0.0059,0.0613,1.0150
2,frozenset({sausage}),frozenset({yogurt}),0.0057,0.0952,1.1090
3,frozenset({yogurt}),frozenset({sausage}),0.0057,0.0669,1.1090
4,frozenset({frankfurter}),frozenset({other vegetables}),0.0051,0.1363,1.1161
